In [ ]:
#install arxiv and wikipedia from requirements.txt
#for now using wikipedia as 1st tool for multisearch agent rag app is wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [2]:
my_api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
tool=WikipediaQueryRun(api_wrapper=my_api_wrapper)

In [4]:
tool.name

'wikipedia'

In [ ]:
##Created 2nd retrieval tool called Ollama_search
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.ollama.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=250).split_documents(docs)
vectordb=FAISS.from_documents(documents,OllamaEmbeddings(model="nomic-embed-text"))
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020A748C1890>, search_kwargs={})

In [11]:
from langchain.tools.retriever import create_retriever_tool
retrieval_tool=create_retriever_tool(retriever,"ollama_search",
                      "Search for information about Ollama. For any queries about Ollama, you must use this tool")

In [12]:
retrieval_tool.name

'ollama_search'

In [ ]:
##created 3rd search tool using ARXIV which has huge research papers records
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

my_arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv=ArxivQueryRun(arxiv_wrapper=my_arxiv_wrapper)
arxiv.name

'arxiv'

In [14]:
tools=[tool,arxiv,retrieval_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\nikhil\\projects\\rag_bot\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 Tool(name='ollama_search', description='Search for information about Ollama. For any queries about Ollama, you must use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000020A74A2BCE0>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorst

In [ ]:
##need to import chat generation model(not text llm model)
from dotenv import load_dotenv

load_dotenv()
import os
os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
from langchain_ollama import ChatOllama

llm=ChatOllama(model='mistral',temperature=0)


In [36]:
##need to create or import prompts
from langchain import hub

prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [37]:
##Agents(use both llm & prompt)
from langchain.agents import create_openai_tools_agent
my_agent=create_openai_tools_agent(llm,tools,prompt)

In [38]:
##Agent executor
from langchain.agents import AgentExecutor

agent_executor=AgentExecutor(agent=my_agent,tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [39]:
agent_executor.invoke({"input":"Tell me about Ollama"})



> Entering new AgentExecutor chain...

Invoking: `ollama_search` with `{'query': 'Ollama'}`


Ollama is the easiest way to get up and running with large language models such as gpt-oss, Gemma 3, DeepSeek-R1, Qwen3 and more.
QuickstartGet up and running with your first model or integrate Ollama with your favorite toolsDownload OllamaDownload Ollama on macOS, Windows or LinuxCloudOllama’s cloud models offer larger models with better performance.API referenceView Ollama’s API reference
​Libraries
Ollama's Python LibraryThe official library for using Ollama with PythonOllama's JavaScript libraryThe official library for using Ollama with JavaScript or TypeScript.Community librariesView a list of 20+ community-supported libraries for Ollama
​Community
DiscordJoin our Discord communityRedditJoin our Reddit communityQuickstartNext⌘IOn this pageLibrariesCommunity

Ollama's documentation - OllamaSkip to main contentOllama home pageSearch...⌘KGet startedWelcomeQuickstartCloudCapabilitiesStreami

{'input': 'Tell me about Ollama',
 'output': ' Ollama is a platform that makes it easy for developers to work with large language models like gpt-oss, Gemma 3, DeepSeek-R1, Qwen3, and others. It provides quickstart guides, libraries for Python and JavaScript, and cloud models for better performance. The platform also has an active community on Discord and Reddit.\n\nTo get started with Ollama, you can download it on macOS, Windows, or Linux, or use its cloud models. The platform offers a variety of capabilities such as streaming, structured outputs, vision, embeddings, web search integration, and more. It also supports various tools like assistants, coding IDEs & editors, chat & RAG, automation notebooks, and more.\n\nOllama provides an API reference for developers to integrate it with their favorite tools. There are also community-supported libraries available for Ollama. For more information, you can check out the CLI reference, assistant sandboxing, model file reference, context len